In [12]:
"""
Tool Wear Prediction using Machine Learning
Intel Unnati: AI for Mechanical Engineers - Group 5
Students: Harsh More (220909420), Samagya Sharma (230909150)
Topic: Tool Wear Prediction
"""

'\nTool Wear Prediction using Machine Learning\nIntel Unnati: AI for Mechanical Engineers - Group 5\nStudents: Harsh More (220909420), Samagya Sharma (230909150)\nTopic: Tool Wear Prediction\n'

# ─────────────────────────────────────────────
# SECTION 0 – LIBRARY IMPORTS
# ─────────────────────────────────────────────

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ─────────────────────────────────────────────
# SECTION 1 – LOAD & EXPLORE DATASET
# ─────────────────────────────────────────────

In [14]:
print("=" * 60)
print("  TOOL WEAR PREDICTION – GROUP 5")
print("=" * 60)

df = pd.read_csv("Tool_wear_dataset.csv")

print("\n[1] Dataset Shape:", df.shape)
print("\n[2] First 5 rows:")
print(df.head())
print("\n[3] Statistical Summary:")
print(df.describe())
print("\n[4] Missing Values:")
print(df.isnull().sum())
print("\n[5] Tool Coating Value Counts:")
print(df["Tool_Coating"].value_counts())

  TOOL WEAR PREDICTION – GROUP 5

[1] Dataset Shape: (91, 8)

[2] First 5 rows:
   Spindle_Speed_RPM  Feed_mm_per_rev  Depth_of_Cut_mm  Vibration_m_s2  \
0               1314            0.103            0.638           3.615   
1               1289            0.159            0.516           3.802   
2               1758            0.142            0.725           4.090   
3               1632            0.134            0.578           4.213   
4               1401            0.160            0.904           4.774   

   Cutting_Force_N  Temperature_C Tool_Coating  Tool_Wear_mm  
0           318.42          56.81          TiN         0.235  
1           328.43          57.89        TiAlN         0.231  
2           365.37          60.95          DLC         0.247  
3           320.02          54.27     Uncoated         0.210  
4           387.93          66.00        TiAlN         0.302  

[3] Statistical Summary:
       Spindle_Speed_RPM  Feed_mm_per_rev  Depth_of_Cut_mm  Vibration_m

# ─────────────────────────────────────────────
# SECTION 2 – EXPLORATORY DATA ANALYSIS (EDA)
# ─────────────────────────────────────────────

In [15]:
print("\n[INFO] Generating EDA plots...")

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Feature Distribution – Tool Wear Dataset", fontsize=14, fontweight="bold")

num_features = ["Spindle_Speed_RPM", "Feed_mm_per_rev", "Depth_of_Cut_mm",
                "Vibration_m_s2", "Cutting_Force_N", "Temperature_C"]

for ax, col in zip(axes.flatten(), num_features):
    ax.hist(df[col], bins=25, color="steelblue", edgecolor="white", alpha=0.85)
    ax.set_title(col)
    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("eda_feature_distributions.png", dpi=150)
plt.close()

# Correlation heat-map
plt.figure(figsize=(9, 7))
corr = df.drop(columns="Tool_Coating").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_correlation_heatmap.png", dpi=150)
plt.close()

# Tool coating vs tool wear
plt.figure(figsize=(8, 5))
df.boxplot(column="Tool_Wear_mm", by="Tool_Coating",
           patch_artist=True, notch=False,
           boxprops=dict(facecolor="lightblue"),
           medianprops=dict(color="red", linewidth=2))
plt.title("Tool Wear by Coating Type")
plt.suptitle("")
plt.xlabel("Tool Coating")
plt.ylabel("Tool Wear (mm)")
plt.tight_layout()
plt.savefig("eda_coating_vs_wear.png", dpi=150)
plt.close()

print("[INFO] EDA plots saved.")


[INFO] Generating EDA plots...
[INFO] EDA plots saved.


<Figure size 800x500 with 0 Axes>

# ─────────────────────────────────────────────
# SECTION 3 – PREPROCESSING
# ─────────────────────────────────────────────

In [16]:
le = LabelEncoder()
df["Tool_Coating_Enc"] = le.fit_transform(df["Tool_Coating"])

feature_cols = ["Spindle_Speed_RPM", "Feed_mm_per_rev", "Depth_of_Cut_mm",
                "Vibration_m_s2", "Cutting_Force_N", "Temperature_C", "Tool_Coating_Enc"]
target_col = "Tool_Wear_mm"

X = df[feature_cols].values
y = df[target_col].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"\n[INFO] Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")


[INFO] Train size: 72 | Test size: 19


# ─────────────────────────────────────────────
# SECTION 4 – MODEL TRAINING & EVALUATION
# ─────────────────────────────────────────────

In [17]:
models = {
    "Linear Regression":       LinearRegression(),
    "Ridge Regression":        Ridge(alpha=1.0),
    "Random Forest":           RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting":       GradientBoostingRegressor(n_estimators=100, random_state=42),
    "Support Vector Regressor": SVR(kernel="rbf", C=10, epsilon=0.01),
}

results = {}
print("\n[INFO] Training models...\n")
print(f"{'Model':<28}  {'MAE':>8}  {'RMSE':>8}  {'R²':>8}  {'Time(s)':>9}")
print("-" * 68)

for name, model in models.items():
    t0 = time.time()
    # Use scaled data for all models for fairness
    model.fit(X_train_sc, y_train)
    elapsed = time.time() - t0

    y_pred = model.predict(X_test_sc)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2, "Time": elapsed, "y_pred": y_pred}
    print(f"{name:<28}  {mae:>8.5f}  {rmse:>8.5f}  {r2:>8.4f}  {elapsed:>9.4f}")


[INFO] Training models...

Model                              MAE      RMSE        R²    Time(s)
--------------------------------------------------------------------
Linear Regression              0.00347   0.00502    0.9794     0.0177
Ridge Regression               0.00348   0.00496    0.9798     0.0037
Random Forest                  0.00245   0.00324    0.9914     0.1357
Gradient Boosting              0.00305   0.00368    0.9889     0.0942
Support Vector Regressor       0.00500   0.00603    0.9702     0.0030


# ─────────────────────────────────────────────
# SECTION 5 – BEST MODEL ANALYSIS
# ─────────────────────────────────────────────

In [18]:
best_name = max(results, key=lambda k: results[k]["R2"])
best      = results[best_name]
print(f"\n[BEST MODEL] {best_name}")
print(f"  MAE  = {best['MAE']:.5f} mm")
print(f"  RMSE = {best['RMSE']:.5f} mm")
print(f"  R²   = {best['R2']:.4f}")

# Actual vs Predicted (best model)
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best["y_pred"], alpha=0.65, color="royalblue", edgecolors="white", s=50)
mn, mx = y_test.min(), y_test.max()
plt.plot([mn, mx], [mn, mx], "r--", linewidth=1.8, label="Ideal")
plt.xlabel("Actual Tool Wear (mm)")
plt.ylabel("Predicted Tool Wear (mm)")
plt.title(f"Actual vs Predicted – {best_name}\n(R² = {best['R2']:.4f})")
plt.legend()
plt.tight_layout()
plt.savefig("results_actual_vs_predicted.png", dpi=150)
plt.close()

# Residual plot
residuals = y_test - best["y_pred"]
plt.figure(figsize=(8, 5))
plt.scatter(best["y_pred"], residuals, alpha=0.65, color="darkorange", edgecolors="white", s=50)
plt.axhline(0, color="black", linewidth=1.2, linestyle="--")
plt.xlabel("Predicted Tool Wear (mm)")
plt.ylabel("Residual (mm)")
plt.title(f"Residual Plot – {best_name}")
plt.tight_layout()
plt.savefig("results_residual_plot.png", dpi=150)
plt.close()

# Model comparison bar chart
metrics_df = pd.DataFrame({
    "Model": list(results.keys()),
    "R²":    [results[k]["R2"]  for k in results],
    "RMSE":  [results[k]["RMSE"] for k in results],
}).sort_values("R²", ascending=False)

fig, ax1 = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_df))
bars = ax1.bar(x - 0.2, metrics_df["R²"], 0.35, label="R²", color="steelblue")
ax2 = ax1.twinx()
ax2.bar(x + 0.2, metrics_df["RMSE"], 0.35, label="RMSE", color="salmon")
ax1.set_xticks(x)
ax1.set_xticklabels(metrics_df["Model"], rotation=15, ha="right")
ax1.set_ylabel("R² Score")
ax2.set_ylabel("RMSE (mm)")
ax1.set_title("Model Comparison – R² and RMSE")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right")
plt.tight_layout()
plt.savefig("results_model_comparison.png", dpi=150)
plt.close()

# Feature importance (Random Forest)
rf_model = models["Random Forest"]
importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    "Feature":    feature_cols,
    "Importance": importances
}).sort_values("Importance", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(feat_imp_df["Feature"], feat_imp_df["Importance"], color="mediumseagreen", edgecolor="white")
plt.xlabel("Feature Importance Score")
plt.title("Feature Importance – Random Forest")
plt.tight_layout()
plt.savefig("results_feature_importance.png", dpi=150)
plt.close()

print("\n[INFO] All result plots saved.")
print("\n" + "=" * 60)
print("  EXECUTION COMPLETE")
print("=" * 60)


[BEST MODEL] Random Forest
  MAE  = 0.00245 mm
  RMSE = 0.00324 mm
  R²   = 0.9914

[INFO] All result plots saved.

  EXECUTION COMPLETE
